## Problem Statement

### Business Context

GlobalEdge Brokerage is a mid-sized brokerage firm operating across multiple countries, serving thousands of retail and institutional clients through a network of equity brokers. Brokers handle investment recommendations, portfolio reviews, and market advisory discussions across equity, forex, commodity, crypto, and international stock markets. Each broker manages hundreds of clients and is expected to stay updated with overnight market developments before the market opens.

Every morning, brokers must review large volumes of financial news, stock-price movements, earnings discussions, analyst commentary, and SEC regulatory filings within a very limited preparation window. In practice, brokers can only read a small portion of the available information before their first client calls begin. As a result, many important signals, disclosures, and market events remain unnoticed.

This creates an intelligence gap during client interactions. Brokers often rely on partial information, memory, or fragmented market sources while answering client questions. Important disclosures buried inside long 10-K and 10-Q filings are difficult to review manually, and brokers may struggle to provide evidence-backed responses when clients ask for justification or supporting references. This increases both compliance risk and trust-related challenges.

To solve this problem, GlobalEdge wants to build a financial intelligence assistant that allows brokers to ask natural-language questions and receive grounded answers backed by real financial data, news articles, and regulatory filings. The system should reduce information overload, improve market coverage, and help brokers make faster and more confident advisory decisions without adding technical complexity to their workflow.

### Objective

This project proposes a proof-of-concept Retrieval-Augmented Generation (RAG) financial intelligence system that combines financial news, stock-price data, and SEC filings into a searchable intelligence layer. The system uses semantic retrieval with a local Chroma vector database to fetch relevant financial context and generate grounded answers for broker questions using large language models. Brokers can ask plain-English questions such as market sentiment analysis, company-risk queries, filing-related questions, or cross-market comparisons and receive evidence-backed responses.

To improve the quality and reliability of generated answers, the system introduces a DeepEval-based prompt optimization workflow using GEPA (Genetic-Pareto Prompt Optimization). Instead of manually refining prompts, the workflow uses benchmark financial question-answer examples and evaluation metrics such as faithfulness, groundedness, relevance, and actionability to optimize the final answering prompt.

The project also evaluates multiple RAG configurations by tuning retrieval and generation parameters such as chunk size, chunk overlap, number of retrieved chunks, temperature, top-p, and maximum token limits. Each configuration is evaluated using DeepEval metrics to identify the most effective combination for grounded financial reasoning and broker-style question answering.

The final system uses the optimized prompt together with the best-performing RAG configuration to generate accurate, grounded, and production-style financial intelligence responses for broker workflows.

### Data Description

The system uses three financial intelligence datasets collected through a custom data ingestion pipeline and stored locally for the RAG workflow.
1. Global Financial News (global_news.csv)
Contains financial news articles with titles, article content, publication dates, URLs, and source metadata. The dataset covers company announcements, market developments, macroeconomic events, sector movements, and sentiment-related signals.
2. Global Stock Prices (all_prices_clean.csv)
Contains historical stock-price data across global equity markets, crypto markets, and forex markets. Each record includes ticker symbols, company names, open/high/low/close prices, trading volume, and timestamps for market analysis and trend evaluation.
3. SEC Regulatory Filings (sec_filings.txt)
Contains regulatory filing documents including 10-K annual reports, 10-Q quarterly reports, and other SEC disclosures. These filings include financial statements, governance information, risk disclosures, operational commentary, and compliance-related information used for grounded financial reasoning.

## Installing and Importing Necessary Libraries and Dependencies

In [9]:
# Objective: install the compatible libraries required through Section 2.4.
# Run this once, restart the kernel as noted below, and then execute sequentially.
%pip install -qU \
    "chromadb==1.5.9" \
    "langchain-community==0.4.1" \
    "langchain-chroma==1.1.0" \
    "langchain-openai==1.2.1" \
    "langchain-text-splitters==1.1.2" \
    "pandas>=2.2.0" \
    "numpy>=1.26.0" \
    "pypdf>=5.0.0" \
    "scikit-learn>=1.4.0" \
    "python-dotenv>=1.0.1" \
    "tqdm>=4.66.0" \
    "deepeval"


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for VSCode), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [10]:
# Importing the necessary libraries
# Objective: centralize every dependency used through Section 2.4 so that
# the notebook execution order and external requirements remain explicit.

# Standard Python libraries for file handling, text processing, JSON handling,
# randomness, timing, and basic utilities.
import os
import re
import json
import random
import time
from pathlib import Path
from collections import Counter
from hashlib import sha256
import pandas as pd
from dotenv import load_dotenv

# DeepEval libraries for prompt optimization and evaluation.
import deepeval
from deepeval.prompt import Prompt
from deepeval.dataset import Golden
from deepeval.metrics import GEval
from deepeval.optimizer import PromptOptimizer
from deepeval.optimizer.algorithms import GEPA
from deepeval.test_case import LLMTestCase, SingleTurnParams
from deepeval.models import GPTModel
from deepeval.optimizer.policies import TieBreaker

# LangChain libraries for document loading, text splitting, embeddings,
# vector storage, and LLM interaction.
import chromadb
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.documents import Document

## Environment Setup and Project Initialization

In [11]:
env_path = Path("../.env") if Path("../.env").exists() else Path(".env")
load_dotenv(env_path)

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY was not found. Add it to your .env file before running the notebook.")

print(f"Loaded environment variables from: {env_path.resolve()}")

Loaded environment variables from: /Users/mbagnara/Desktop/UofTexas/grounded-investment-assistant/.env


# Section 1: Baseline Retrieval-Augmented Generation (RAG) Pipeline

## 1.1 Load the CSV files with `CSVLoader`

In [12]:
# Objective: build explicit LangChain Documents without changing the raw CSVs.
# page_content is optimized for embeddings, while metadata supports filtering,
# traceability, citations, and deterministic retrieval diagnostics.
data_dir = Path("../data/raw") if Path("../data/raw").exists() else Path("data/raw")

stock_prices_path = data_dir / "stock_price_details.csv"
global_news_path = data_dir / "global_news.csv"
stock_df = pd.read_csv(stock_prices_path).fillna("")
news_df = pd.read_csv(global_news_path).fillna("")

price_record_pattern = re.compile(
    r"Date is (?P<date>[^,]+), ticker is (?P<ticker>[^,]+), "
    r"name is (?P<name>[^,]+),"
)

stock_docs = []
unparsed_price_rows = []
for row_id, row in stock_df.iterrows():
    price_summary = str(row["price_summary"]).strip()
    match = price_record_pattern.search(price_summary)
    if not match:
        unparsed_price_rows.append(row_id)
        continue

    fields = match.groupdict()
    stock_docs.append(Document(
        page_content=price_summary,
        metadata={
            "dataset": "stock_price_details",
            "source_file": stock_prices_path.name,
            "row_id": int(row_id),
            "ticker": fields["ticker"].strip().upper(),
            "date": fields["date"].strip(),
            "name": fields["name"].strip(),
        },
    ))

if unparsed_price_rows:
    raise ValueError(f"Could not parse stock rows: {unparsed_price_rows[:10]}")

news_docs = []
for row_id, row in news_df.iterrows():
    news_content = f"""
Title: {row['title']}
Published at: {row['published_at']}
Source: {row['source']}

Description:
{row['description']}

Content:
{row['content']}
""".strip()

    news_docs.append(Document(
        page_content=news_content,
        metadata={
            "dataset": "global_news",
            "source_file": global_news_path.name,
            "row_id": int(row_id),
            "title": str(row["title"]),
            "source": str(row["source"]),
            "url": str(row["url"]),
            "published_at": str(row["published_at"]),
            "query": str(row["query"]),
        },
    ))

csv_docs = stock_docs + news_docs

print(f"Loaded {len(stock_docs)} stock price documents")
print(f"Loaded {len(news_docs)} news documents")
print(f"Loaded {len(csv_docs)} total CSV documents")
print("Sample metadata:", csv_docs[0].metadata)
print(csv_docs[0].page_content[:500])

Loaded 500 stock price documents
Loaded 500 news documents
Loaded 1000 total CSV documents
Sample metadata: {'dataset': 'stock_price_details', 'source_file': 'stock_price_details.csv', 'row_id': 0, 'ticker': '0700.HK', 'date': '2026-03-12', 'name': 'Tencent (Hong Kong)'}
On this record, Date is 2026-03-12, ticker is 0700.HK, name is Tencent (Hong Kong), Open is 550.5, High is 557.0, Low is 541.5, Close is 546.5, Volume is 19663840.0.


## 1.2 Load the SEC filings text and split it into chunks


In [13]:
# Objective: preserve page-level provenance while creating embedding-sized
# SEC chunks. The PDF remains unchanged; only in-memory metadata is enriched.
sec_pdf_path = data_dir / "sec_filings_10q.pdf"

if not sec_pdf_path.exists():
    raise FileNotFoundError("Expected sec_filings_10q.pdf in the raw data folder.")

sec_loader = PyPDFLoader(str(sec_pdf_path))

sec_raw_docs = sec_loader.load()

if not sec_raw_docs:
    raise ValueError("The SEC PDF did not produce any page documents.")

for doc in sec_raw_docs:
    doc.metadata.update({
        "dataset": "sec_filings",
        "source_file": sec_pdf_path.name,
        "document_type": "10-Q",
    })

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)

sec_docs = text_splitter.split_documents(sec_raw_docs)
for chunk_id, doc in enumerate(sec_docs):
    doc.metadata["chunk_id"] = chunk_id

if not sec_docs or any(not doc.page_content.strip() for doc in sec_docs):
    raise ValueError("SEC chunking produced empty content.")
all_docs = csv_docs + sec_docs

print(f"Loaded {len(sec_raw_docs)} SEC filing source documents")
print(f"Created {len(sec_docs)} SEC filing chunks")
print(f"Prepared {len(all_docs)} total documents for vector indexing")
print(sec_docs[0].page_content[:500])

Loaded 126 SEC filing source documents
Created 522 SEC filing chunks
Prepared 1522 total documents for vector indexing
UNITED STATES
SECURITIES AND EXCHANGE COMMISSION
Washington, D.C. 20549
________________________________________________________________________________________
FORM 10-Q
_______________________________________________________________________________________
(Mark One)
☒ QUARTERLY REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the quarterly period ended March 31, 2026OR
☐ TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
F


## 1.3 Build a local Chroma vector store

In [14]:
# Objective: rebuild a deterministic Version 2 collection on every run.
# This prevents duplicate documents from changing top-k results after a cell rerun.
chroma_dir = Path("../chroma_db") if Path("../data/raw").exists() else Path("chroma_db")
collection_name = "investment_rag_v2"
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

chroma_client = chromadb.PersistentClient(path=str(chroma_dir))
try:
    chroma_client.delete_collection(collection_name)
except Exception:
    # A missing collection is expected on the first run.
    pass

def deterministic_document_id(doc):
    identity = json.dumps(doc.metadata, sort_keys=True) + doc.page_content
    return sha256(identity.encode("utf-8")).hexdigest()

document_ids = [deterministic_document_id(doc) for doc in all_docs]
if len(document_ids) != len(set(document_ids)):
    raise ValueError("Duplicate document identities were detected before indexing.")

vector_store = Chroma(
    client=chroma_client,
    collection_name=collection_name,
    embedding_function=embeddings,
)
vector_store.add_documents(all_docs, ids=document_ids)
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5},
)

dataset_counts = Counter(doc.metadata["dataset"] for doc in all_docs)
print(f"Created Chroma collection: {collection_name}")
print(f"Persisted vector store at: {chroma_dir.resolve()}")
print(f"Indexed {vector_store._collection.count()} documents")
print("Documents by dataset:", dict(dataset_counts))

Created Chroma collection: investment_rag_v2
Persisted vector store at: /Users/mbagnara/Desktop/UofTexas/grounded-investment-assistant/chroma_db
Indexed 1522 documents
Documents by dataset: {'stock_price_details': 500, 'global_news': 500, 'sec_filings': 522}


## 1.4 Retrieve Context and Generate Answers using the Baseline RAG Pipeline

In [15]:
# Objective: make retrieval observable and source-aware while preserving a
# transparent similarity-search baseline. Routing uses only the user question.
baseline_system_prompt = """
You are a grounded investment intelligence assistant for brokerage analysts.
Use only the retrieved context to answer the user's question.
If the context is insufficient, say what is missing instead of guessing.
Keep the answer concise, evidence-backed, and suitable for a broker speaking with a client.
""".strip()


def format_source_metadata(metadata):
    source_file = metadata.get("source_file") or metadata.get("source") or metadata.get("dataset", "unknown")
    title = metadata.get("title")
    published_at = metadata.get("published_at")
    page = metadata.get("page")
    row = metadata.get("row_id", metadata.get("row"))

    details = [str(source_file)]
    if title:
        details.append(str(title))
    if published_at:
        details.append(str(published_at))
    if page is not None:
        details.append(f"page {page}")
    if row is not None:
        details.append(f"row {row}")

    return " | ".join(details)


def format_retrieved_context(docs):
    formatted_sources = []

    for idx, doc in enumerate(docs, start=1):
        metadata = doc.metadata or {}
        source_label = format_source_metadata(metadata)
        content = re.sub(r"\s+", " ", doc.page_content).strip()
        formatted_sources.append(f"[Source {idx}] {content}\nMetadata: {source_label}")

    return "\n\n".join(formatted_sources)


PRICE_TERMS = {"price", "close", "closing", "open", "high", "low", "volume", "trading volume", "exchange rate"}
SEC_TERMS = {"10-k", "10-q", "sec", "filing", "auditor", "audit", "internal control", "material weakness", "financial statements"}
NEWS_TERMS = {"news", "recent reports", "market chatter", "announced", "outlook", "blockade", "recently", "geopolitical", "exchange and ticker"}
COMPANY_TICKER_ALIASES = {"apple": "AAPL", "tesla": "TSLA", "tencent": "0700.HK", "usd/jpy": "USDJPY=X", "usdjpy": "USDJPY=X"}
KNOWN_TICKERS = sorted({doc.metadata["ticker"] for doc in stock_docs}, key=len, reverse=True)

def route_question(question):
    text = question.lower()
    datasets = []
    if any(term in text for term in PRICE_TERMS):
        datasets.append("stock_price_details")
    if any(term in text for term in SEC_TERMS):
        datasets.append("sec_filings")
    if any(term in text for term in NEWS_TERMS):
        datasets.append("global_news")
    return datasets or ["global_news", "stock_price_details", "sec_filings"]

def extract_question_constraints(question):
    text = question.lower()
    ticker = next((ticker for ticker in KNOWN_TICKERS if ticker.lower() in text), None)
    if ticker is None:
        ticker = next((value for alias, value in COMPANY_TICKER_ALIASES.items() if alias in text), None)
    dates = re.findall(r"\b\d{4}-\d{2}-\d{2}\b", question)
    return {"ticker": ticker, "dates": dates}

def chroma_filter(dataset, ticker=None, date=None):
    conditions = [{"dataset": {"$eq": dataset}}]
    if dataset == "stock_price_details" and ticker:
        conditions.append({"ticker": {"$eq": ticker}})
    if dataset == "stock_price_details" and date:
        conditions.append({"date": {"$eq": date}})
    return conditions[0] if len(conditions) == 1 else {"$and": conditions}

def retrieve_context(question, k=5, search_type="similarity", use_routing=True):
    datasets = route_question(question) if use_routing else ["global_news", "stock_price_details", "sec_filings"]
    constraints = extract_question_constraints(question)
    k_per_dataset = k if len(datasets) == 1 else max(2, k // len(datasets))
    retrieved_docs = []
    searches = []

    for dataset in datasets:
        requested_date = constraints["dates"][0] if len(constraints["dates"]) == 1 else None
        where = chroma_filter(dataset, ticker=constraints["ticker"], date=requested_date)
        if search_type == "mmr":
            docs = vector_store.max_marginal_relevance_search(question, k=k_per_dataset, fetch_k=max(20, k_per_dataset * 4), filter=where)
        else:
            docs = vector_store.similarity_search(question, k=k_per_dataset, filter=where)

        fallback_used = False
        if not docs and dataset == "stock_price_details" and requested_date:
            fallback_used = True
            where = chroma_filter(dataset, ticker=constraints["ticker"])
            docs = vector_store.similarity_search(question, k=k_per_dataset, filter=where)

        retrieved_docs.extend(docs)
        searches.append({"dataset": dataset, "filter": where, "result_count": len(docs), "fallback_used": fallback_used})

    unique_docs = []
    seen_ids = set()
    for doc in retrieved_docs:
        doc_id = deterministic_document_id(doc)
        if doc_id not in seen_ids:
            seen_ids.add(doc_id)
            unique_docs.append(doc)

    diagnostics = {
        "datasets_requested": datasets,
        "constraints": constraints,
        "search_type": search_type,
        "searches": searches,
        "retrieved_count": len(unique_docs),
        "retrieved_datasets": sorted({doc.metadata["dataset"] for doc in unique_docs}),
    }
    return {"documents": unique_docs, "diagnostics": diagnostics}


baseline_llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

def baseline_rag_answer(question, k=5):
    retrieval_result = retrieve_context(question, k=k)
    retrieved_docs = retrieval_result["documents"]
    context = format_retrieved_context(retrieved_docs)

    user_prompt = f"""
Question:
{question}

Retrieved context:
{context}

Answer with a direct response followed by a short Sources section naming the most relevant source numbers.
""".strip()

    response = baseline_llm.invoke([
        SystemMessage(content=baseline_system_prompt),
        HumanMessage(content=user_prompt),
    ])

    return {
        "question": question,
        "answer": response.content,
        "context": context,
        "source_documents": retrieved_docs,
        "retrieval_diagnostics": retrieval_result["diagnostics"],
    }


sample_question = "What are the main risks or signals a broker should discuss with a client considering Apple?"
baseline_sample = baseline_rag_answer(sample_question)

print("Question:")
print(baseline_sample["question"])
print("\nAnswer:")
print(baseline_sample["answer"])
print("\nRetrieved sources:")
for idx, doc in enumerate(baseline_sample["source_documents"], start=1):
    print(f"[{idx}] {format_source_metadata(doc.metadata or {})}")

Question:
What are the main risks or signals a broker should discuss with a client considering Apple?

Answer:
Key risks and signals to discuss with a client considering Apple include:

1. Leadership Change: The recent stepping down of CEO Tim Cook has caused some market uncertainty, as reflected by a nearly 1% premarket drop in Apple's stock. Leadership transitions can impact company strategy and investor confidence.

2. Gross Margin Volatility: Apple expects its gross margins to face volatility and downward pressure, which could affect profitability.

3. Operational and Regulatory Risks: Changes in products, services, or operations may lead to disruptions, increased privacy and data security risks, higher costs, and potential liabilities or fines, all of which could materially impact Apple's business and stock price.

These factors suggest that while Apple remains a strong company, investors should be mindful of leadership shifts, margin pressures, and regulatory/operational challeng

## 1.5 Load the gold benchmark dataset and evaluate the baseline prompt


In [16]:
# Objective: create a leakage-resistant benchmark workflow. GEPA optimization
# and final prompt evaluation use separate, source-balanced subsets. Retrieval
# is executed once and cached so both prompts receive identical evidence.
gold_dataset_path = Path("../data/eval/golden_benchmark_dataset.csv")
if not gold_dataset_path.exists():
    gold_dataset_path = Path("data/eval/golden_benchmark_dataset.csv")

gold_df = pd.read_csv(gold_dataset_path)

required_columns = {"question", "response", "source_hint", "supporting_sources", "context"}
missing_columns = required_columns.difference(gold_df.columns)
if missing_columns:
    raise ValueError(f"Gold benchmark dataset is missing columns: {sorted(missing_columns)}")

gold_df = gold_df.dropna(subset=["question", "response"]).reset_index(drop=True)
gold_df["case_id"] = gold_df.index + 1

SOURCE_DATASET_MAP = {
    "global_news.csv": "global_news",
    "all_prices_clean.csv": "stock_price_details",
    "stock_price_details.csv": "stock_price_details",
    "sec_filings.txt": "sec_filings",
    "sec_filings_10q.pdf": "sec_filings",
}
gold_df["expected_dataset"] = gold_df["supporting_sources"].map(SOURCE_DATASET_MAP)
if gold_df["expected_dataset"].isna().any():
    unknown_sources = sorted(gold_df.loc[gold_df["expected_dataset"].isna(), "supporting_sources"].unique())
    raise ValueError(f"Unknown supporting source labels: {unknown_sources}")

# Select approximately 40% from every expected source for holdout evaluation.
evaluation_df = (
    gold_df.groupby("expected_dataset", group_keys=False)
    .sample(frac=0.4, random_state=42)
    .sort_values("case_id")
    .reset_index(drop=True)
)
optimization_df = (
    gold_df[~gold_df["case_id"].isin(evaluation_df["case_id"])]
    .sort_values("case_id")
    .reset_index(drop=True)
)

benchmark_contexts = {}
for _, row in gold_df.iterrows():
    retrieval_result = retrieve_context(row["question"], k=5)
    docs = retrieval_result["documents"]
    benchmark_contexts[row["question"]] = {
        "documents": docs,
        "formatted_context": format_retrieved_context(docs),
        "retrieval_context": [doc.page_content for doc in docs],
        "retrieved_sources": [format_source_metadata(doc.metadata or {}) for doc in docs],
        "diagnostics": retrieval_result["diagnostics"],
    }

def retrieval_quality(row, cached_context):
    docs = cached_context["documents"]
    retrieved_datasets = sorted({doc.metadata["dataset"] for doc in docs})
    expected_dataset = row["expected_dataset"]
    constraints = extract_question_constraints(row["question"])
    source_match = expected_dataset in retrieved_datasets
    ticker_match = (
        None if not constraints["ticker"] else
        any(doc.metadata.get("ticker") == constraints["ticker"] for doc in docs)
    )
    requested_dates = constraints["dates"]
    date_match = (
        None if not requested_dates else
        all(any(doc.metadata.get("date") == date for doc in docs) for date in requested_dates)
    )
    if not source_match:
        status = "UNSUPPORTED"
    elif ticker_match is False or date_match is False:
        status = "PARTIAL"
    else:
        status = "SUPPORTED"
    return {
        "expected_dataset": expected_dataset,
        "retrieved_datasets": retrieved_datasets,
        "source_match": source_match,
        "ticker_match": ticker_match,
        "date_match": date_match,
        "unique_sources": len(set(cached_context["retrieved_sources"])),
        "coverage_status": status,
    }

baseline_eval_df = evaluation_df.copy()

baseline_rows = []
baseline_test_cases = []

for _, row in baseline_eval_df.iterrows():
    question = row["question"]
    expected_answer = row["response"]
    cached = benchmark_contexts[question]
    retrieval_checks = retrieval_quality(row, cached)

    user_prompt = f"""
Question:
{question}

Retrieved context:
{cached['formatted_context']}

Answer with a direct response followed by a short Sources section naming the most relevant source numbers.
""".strip()
    response = baseline_llm.invoke([
        SystemMessage(content=baseline_system_prompt),
        HumanMessage(content=user_prompt),
    ])

    baseline_rows.append({
        "case_id": int(row["case_id"]),
        "question": question,
        "expected_output": expected_answer,
        "actual_output": response.content,
        "retrieved_context": cached["formatted_context"],
        "retrieved_sources": cached["retrieved_sources"],
        "source_hint": row["source_hint"],
        "supporting_sources": row["supporting_sources"],
        **retrieval_checks,
    })

    baseline_test_cases.append(
        LLMTestCase(
            input=question,
            actual_output=response.content,
            expected_output=expected_answer,
            retrieval_context=cached["retrieval_context"],
        )
    )

baseline_results_df = pd.DataFrame(baseline_rows)

print(f"Loaded {len(gold_df)} gold benchmark examples from {gold_dataset_path.resolve()}")
print(f"Optimization cases: {len(optimization_df)} | Holdout evaluation cases: {len(evaluation_df)}")
print(f"Generated baseline outputs for {len(baseline_results_df)} holdout examples")
display(baseline_results_df[["case_id", "expected_dataset", "retrieved_datasets", "coverage_status", "source_match"]])

Loaded 20 gold benchmark examples from /Users/mbagnara/Desktop/UofTexas/grounded-investment-assistant/data/eval/golden_benchmark_dataset.csv
Optimization cases: 12 | Holdout evaluation cases: 8
Generated baseline outputs for 8 holdout examples


,case_id,expected_dataset,retrieved_datasets,coverage_status,source_match
0,1,sec_filings,[sec_filings],PARTIAL,True
1,4,sec_filings,[sec_filings],PARTIAL,True
2,7,global_news,[global_news],SUPPORTED,True
3,8,global_news,[global_news],SUPPORTED,True
4,13,stock_price_details,[stock_price_details],SUPPORTED,True
5,16,global_news,[global_news],SUPPORTED,True
6,17,sec_filings,"[global_news, sec_filings, stock_price_details]",SUPPORTED,True
7,20,sec_filings,"[sec_filings, stock_price_details]",SUPPORTED,True


## 1.6 Define the evaluation metrics

In [17]:
# Objective: evaluate generation quality with explicit, reproducible success
# thresholds. Retrieval diagnostics remain deterministic Python checks in 1.5.
EVALUATION_THRESHOLD = 0.5

correctness_metric = GEval(
    name="Answer Correctness",
    threshold=EVALUATION_THRESHOLD,
    criteria=(
        "Evaluate whether the actual output correctly answers the input question "
        "and matches the expected output from the gold benchmark dataset."
    ),
    evaluation_params=[
        SingleTurnParams.INPUT,
        SingleTurnParams.ACTUAL_OUTPUT,
        SingleTurnParams.EXPECTED_OUTPUT,
    ],
    async_mode=False,
)

faithfulness_metric = GEval(
    name="Faithfulness / Groundedness",
    threshold=EVALUATION_THRESHOLD,
    criteria=(
        "Evaluate whether the actual output is fully supported by the retrieved context. "
        "Penalize unsupported claims, hallucinations, or facts not present in the context."
    ),
    evaluation_params=[
        SingleTurnParams.ACTUAL_OUTPUT,
        SingleTurnParams.RETRIEVAL_CONTEXT,
    ],
    async_mode=False,
)

context_relevance_metric = GEval(
    name="Context Relevance",
    threshold=EVALUATION_THRESHOLD,
    criteria=(
        "Evaluate whether the retrieved context is relevant and sufficient to answer "
        "the input question."
    ),
    evaluation_params=[
        SingleTurnParams.INPUT,
        SingleTurnParams.RETRIEVAL_CONTEXT,
    ],
    async_mode=False,
)

answer_relevance_metric = GEval(
    name="Answer Relevance",
    threshold=EVALUATION_THRESHOLD,
    criteria=(
        "Evaluate whether the actual output directly answers the input question "
        "without unnecessary or unrelated information."
    ),
    evaluation_params=[
        SingleTurnParams.INPUT,
        SingleTurnParams.ACTUAL_OUTPUT,
    ],
    async_mode=False,
)

broker_actionability_metric = GEval(
    name="Broker Actionability",
    threshold=EVALUATION_THRESHOLD,
    criteria=(
        "Evaluate whether the answer is useful for a broker speaking with a client. "
        "The answer should highlight relevant risks, signals, evidence, and practical implications."
    ),
    evaluation_params=[
        SingleTurnParams.INPUT,
        SingleTurnParams.ACTUAL_OUTPUT,
        SingleTurnParams.RETRIEVAL_CONTEXT,
    ],
    async_mode=False,
)

baseline_metrics = [
    correctness_metric,
    faithfulness_metric,
    context_relevance_metric,
    answer_relevance_metric,
    broker_actionability_metric,
]

# Context Relevance measures retrieval rather than prompt quality, so GEPA
# does not optimize against it. It remains in the final evaluation suite.
prompt_optimization_metrics = [
    correctness_metric,
    faithfulness_metric,
    answer_relevance_metric,
    broker_actionability_metric,
]

## 1.7 Evaluate the baseline RAG on the golden examples dataset

In [18]:
# Objective: score the baseline on the holdout set and retain both long-form
# metric details and a compact one-row-per-case view for notebook readability.
baseline_evaluation_rows = []

for baseline_row, test_case in zip(baseline_rows, baseline_test_cases):
    case_id = baseline_row["case_id"]
    print(f"Evaluating baseline case {case_id} ({len(baseline_evaluation_rows) // len(baseline_metrics) + 1}/{len(baseline_test_cases)})")

    for metric in baseline_metrics:
        try:
            metric.measure(test_case, _show_indicator=False)
            score = metric.score
            reason = metric.reason
            success = metric.is_successful() if hasattr(metric, "is_successful") else None
            error = None
        except Exception as exc:
            score = None
            reason = None
            success = False
            error = str(exc)

        baseline_evaluation_rows.append({
            "case_id": case_id,
            "question": test_case.input,
            "metric": metric.name,
            "score": score,
            "success": success,
            "reason": reason,
            "error": error,
            "actual_output": test_case.actual_output,
            "expected_output": test_case.expected_output,
            "coverage_status": baseline_row["coverage_status"],
            "source_match": baseline_row["source_match"],
        })

baseline_evaluation_df = pd.DataFrame(baseline_evaluation_rows)

baseline_metric_summary_df = (
    baseline_evaluation_df
    .groupby("metric", dropna=False)
    .agg(
        average_score=("score", "mean"),
        evaluated_cases=("score", "count"),
        successful_cases=("success", "sum"),
        failed_or_error_cases=("error", lambda values: values.notna().sum()),
    )
    .reset_index()
    .sort_values("average_score", ascending=False)
)

print("Baseline RAG evaluation summary")
display(baseline_metric_summary_df)

baseline_scores_pivot_df = baseline_evaluation_df.pivot(
    index="case_id", columns="metric", values="score"
).reset_index()

print("Baseline scores by holdout case")
display(baseline_scores_pivot_df)
print("Retrieval coverage summary")
display(baseline_results_df["coverage_status"].value_counts().rename_axis("status").reset_index(name="cases"))

Evaluating baseline case 1 (1/8)
Evaluating baseline case 4 (2/8)
Evaluating baseline case 7 (3/8)
Evaluating baseline case 8 (4/8)
Evaluating baseline case 13 (5/8)
Evaluating baseline case 16 (6/8)
Evaluating baseline case 17 (7/8)
Evaluating baseline case 20 (8/8)
Baseline RAG evaluation summary


,metric,average_score,evaluated_cases,successful_cases,failed_or_error_cases
1,Answer Relevance,0.856288,8,7,0
2,Broker Actionability,0.795188,8,7,0
4,Faithfulness / Groundedness,0.704607,8,6,0
0,Answer Correctness,0.658159,8,6,0
3,Context Relevance,0.481362,8,3,0


Baseline scores by holdout case


metric,case_id,Answer Correctness,Answer Relevance,Broker Actionability,Context Relevance,Faithfulness / Groundedness
0,1,0.895257,1.000000,0.268795,0.202298,0.257509
1,4,0.689194,0.901799,0.683629,0.349327,0.642702
2,7,0.783121,1.000000,1.000000,0.932082,0.905345
3,8,1.000000,1.000000,1.000000,0.993991,0.470066
4,13,0.081757,0.768205,0.922270,0.185195,0.981757
5,16,0.997702,0.997069,0.796267,0.811920,0.723259
6,17,0.818243,0.846357,0.785195,0.294322,0.700000
7,20,0.000000,0.336873,0.905345,0.081757,0.956218


Retrieval coverage summary


,status,cases
0,SUPPORTED,6
1,PARTIAL,2


# Section 2: DeepEval Prompt Optimization Layer

## 2.1 Define the prompt template and model callback


In [19]:
# Objective: optimize only the prompt. Every candidate receives a frozen
# cached context, so retrieval cannot confound the GEPA comparison.
optimized_prompt_seed = Prompt(
    text_template="""
You are a grounded investment intelligence assistant for brokerage analysts.

Use only the retrieved context to answer the question. Do not invent facts.
If the context does not contain enough evidence, clearly state what is missing.
Write a concise client-ready answer that explains the relevant financial signal,
risk, or implication. Reference the most relevant source numbers when useful.

Question:
{question}

Retrieved context:
{context}

Answer:
""".strip()
)
seed_prompt_text = optimized_prompt_seed.text_template

prompt_optimization_goldens = [
    Golden(
        input=row["question"],
        expected_output=row["response"],
        additional_metadata={
            "source_hint": row["source_hint"],
            "supporting_sources": row["supporting_sources"],
            "case_id": int(row["case_id"]),
        },
    )
    for _, row in optimization_df.iterrows()
]


def optimized_prompt_model_callback(prompt, golden):
    question = golden.input
    cached = benchmark_contexts[question]
    golden.retrieval_context = cached["retrieval_context"]

    rendered_prompt = prompt.interpolate(
        question=question,
        context=cached["formatted_context"],
    )

    llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)
    response = llm.invoke([HumanMessage(content=rendered_prompt)])

    return response.content


print(f"Prepared {len(prompt_optimization_goldens)} goldens for prompt optimization")
print("Seed prompt preview:")
print(optimized_prompt_seed.text_template[:700])

Prepared 12 goldens for prompt optimization
Seed prompt preview:
You are a grounded investment intelligence assistant for brokerage analysts.

Use only the retrieved context to answer the question. Do not invent facts.
If the context does not contain enough evidence, clearly state what is missing.
Write a concise client-ready answer that explains the relevant financial signal,
risk, or implication. Reference the most relevant source numbers when useful.

Question:
{question}

Retrieved context:
{context}

Answer:


## 2.2 Optimize the prompt with GEPA

In [20]:
# Objective: run a low-cost, reproducible GEPA optimization using only the
# optimization split. The holdout cases remain unseen until Section 2.3.
from deepeval.optimizer.configs import AsyncConfig, DisplayConfig
from deepeval.optimizer.scorer.scorer import Scorer
from deepeval.optimizer.rewriter.rewriter import Rewriter

print(f"DeepEval version: {getattr(deepeval, '__version__', 'unknown')}")

# Workaround for a DeepEval GEPA token-accounting bug in this installed version.
# GEPA calls _accrue_tokens on helper classes that do not define it.
def _optimizer_helper_accrue_tokens(self, input_tokens=0, output_tokens=0):
    self.input_tokens = getattr(self, "input_tokens", 0) + (input_tokens or 0)
    self.output_tokens = getattr(self, "output_tokens", 0) + (output_tokens or 0)


for helper_cls in (Scorer, Rewriter):
    if not hasattr(helper_cls, "_accrue_tokens"):
        helper_cls._accrue_tokens = _optimizer_helper_accrue_tokens

gepa_algorithm = GEPA(
    iterations=3,
    minibatch_size=8,
    pareto_size=5,
    patience=2,
    random_seed=42,
    tie_breaker=TieBreaker.PREFER_CHILD,
    reflection_model="gpt-4.1-mini",
    mutation_model="gpt-4.1-mini",
)

prompt_optimizer = PromptOptimizer(
    model_callback=optimized_prompt_model_callback,
    metrics=prompt_optimization_metrics,
    optimizer_model="gpt-4.1-mini",
    algorithm=gepa_algorithm,
    async_config=AsyncConfig(run_async=False, max_concurrent=1, throttle_value=0.5),
    display_config=DisplayConfig(show_indicator=False),
)

optimized_prompt = prompt_optimizer.optimize(
    prompt=optimized_prompt_seed,
    goldens=prompt_optimization_goldens,
)

optimization_report = prompt_optimizer.optimization_report
optimized_prompt_text = optimized_prompt.text_template

print("Optimized prompt:")
print(optimized_prompt_text)


DeepEval version: 4.0.9
Optimized prompt:
You are a grounded investment intelligence assistant for brokerage analysts.

Use only the retrieved context to answer the question. Do not invent facts, speculate, or infer information beyond what is explicitly stated in the context.
If the context does not contain enough evidence to answer precisely, clearly state what is missing without adding assumptions.
Avoid introducing unrelated temporal or contextual details not present in the retrieved context.
Write a concise client-ready answer that explains the relevant financial signal, risk, or implication.
Reference the most relevant source numbers when useful.

Question:
{question}

Retrieved context:
{context}

Answer:


## 2.3 Evaluate the optimized prompt on the benchmark dataset

In [21]:
# Objective: evaluate the optimized prompt only on the holdout split. The
# cached baseline contexts are reused exactly, isolating the prompt as the
# single experimental variable.
optimized_llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

# Generate answers with the optimized prompt using the same benchmark and
# retrieval settings used for the baseline evaluation.
optimized_rows = []
optimized_test_cases = []

for _, row in evaluation_df.iterrows():
    question = row["question"]
    expected_answer = row["response"]
    cached = benchmark_contexts[question]
    retrieval_checks = retrieval_quality(row, cached)

    rendered_prompt = optimized_prompt.interpolate(
        question=question,
        context=cached["formatted_context"],
    )

    response = optimized_llm.invoke([
        HumanMessage(content=rendered_prompt)
    ])
    actual_answer = response.content

    optimized_rows.append({
        "case_id": int(row["case_id"]),
        "question": question,
        "expected_output": expected_answer,
        "actual_output": actual_answer,
        "retrieved_context": cached["formatted_context"],
        "retrieved_sources": cached["retrieved_sources"],
        "source_hint": row["source_hint"],
        "supporting_sources": row["supporting_sources"],
        **retrieval_checks,
    })

    optimized_test_cases.append(
        LLMTestCase(
            input=question,
            actual_output=actual_answer,
            expected_output=expected_answer,
            retrieval_context=cached["retrieval_context"],
        )
    )

optimized_results_df = pd.DataFrame(optimized_rows)

print(f"Generated optimized-prompt outputs for {len(optimized_results_df)} examples")
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", 300)
display(
    optimized_results_df[
        ["question", "expected_output", "actual_output", "supporting_sources"]
    ]
)


# Evaluate the optimized answers with the same metrics used for the baseline.
optimized_evaluation_rows = []

for optimized_row, test_case in zip(optimized_rows, optimized_test_cases):
    case_id = optimized_row["case_id"]
    print(
        f"Evaluating optimized-prompt case {case_id} "
        f"({len(optimized_evaluation_rows) // len(baseline_metrics) + 1}/{len(optimized_test_cases)})"
    )

    for metric in baseline_metrics:
        try:
            metric.measure(test_case, _show_indicator=False)
            score = metric.score
            reason = metric.reason
            success = (
                metric.is_successful()
                if hasattr(metric, "is_successful")
                else None
            )
            error = None
        except Exception as exc:
            score = None
            reason = None
            success = False
            error = str(exc)

        optimized_evaluation_rows.append({
            "case_id": case_id,
            "question": test_case.input,
            "metric": metric.name,
            "score": score,
            "success": success,
            "reason": reason,
            "error": error,
            "actual_output": test_case.actual_output,
            "expected_output": test_case.expected_output,
            "coverage_status": optimized_row["coverage_status"],
            "source_match": optimized_row["source_match"],
        })

optimized_evaluation_df = pd.DataFrame(optimized_evaluation_rows)

optimized_metric_summary_df = (
    optimized_evaluation_df
    .groupby("metric", dropna=False)
    .agg(
        average_score=("score", "mean"),
        evaluated_cases=("score", "count"),
        successful_cases=("success", "sum"),
        failed_or_error_cases=(
            "error",
            lambda values: values.notna().sum(),
        ),
    )
    .reset_index()
    .sort_values("average_score", ascending=False)
)

print("Optimized prompt evaluation summary")
display(optimized_metric_summary_df)

# Present one row per benchmark case, with each metric as a column.
# The full long-form results remain available in optimized_evaluation_df.
optimized_scores_pivot_df = optimized_evaluation_df.pivot(
    index="case_id",
    columns="metric",
    values="score",
).reset_index()

print("Optimized prompt scores by benchmark case")
display(optimized_scores_pivot_df)


Generated optimized-prompt outputs for 8 examples


,question,expected_output,actual_output,supporting_sources
0,"Given the internal control audit for the fiscal year ended September 27, 2025, did Apple’s auditors identify any material weaknesses, and how does this affect the reliability of the current financial disclosures?","No material weaknesses were identified. Ernst & Young LLP issued an unqualified opinion stating that Apple maintained effective internal control over financial reporting as of September 27, 2025 based on the COSO 2013 framework. This supports a high level of reliability for the company’s financi...","Apple’s auditors did not identify any material weaknesses in internal control over financial reporting for the fiscal year ended September 27, 2025. Management, including the principal executive and financial officers, concluded that the Company’s disclosure controls and procedures were effectiv...",sec_filings.txt
1,A client is skeptical of automated financial reporting. How does Apple define Internal Control Over Financial Reporting and what limitation does management acknowledge?,"Apple states that internal controls are designed to provide reasonable assurance regarding financial reporting reliability. Management also acknowledges that internal controls cannot prevent or detect all errors and fraud, and therefore provide reasonable rather than absolute assurance.","Apple defines Internal Control Over Financial Reporting as a system of disclosure controls and procedures designed to provide reasonable assurance that information required to be disclosed in reports filed under the Exchange Act is recorded, processed, summarized, and reported within the require...",sec_filings.txt
2,SRx Health Solutions recently announced a pivot toward the convergence of AI and space. What is the scale of the investment and who is their digital-asset partner?,SRx Health Solutions invested more than 10 percent of its investable capital into Astro Investment XVII and partnered with EMJ Crypto Technologies as the digital-asset treasury operating platform.,"SRx Health Solutions has committed more than 10% of its investable capital toward the convergence of Artificial Intelligence (AI) and space through an investment in Astro Investment XVII, an affiliate of Astro Capital. Their digital-asset partner in this initiative is EMJ Crypto Technologies (EM...",global_news.csv
3,"For brokers monitoring small-cap AI plays, what exchange and ticker symbol should be tracked for SRx Health Solutions?",SRx Health Solutions trades on the NYSE American exchange under the ticker SRXH.,Brokers monitoring small-cap AI plays should track SRx Health Solutions on the NYSE American exchange under the ticker symbol **SRXH** (Source 1).,global_news.csv
4,"Compare Tencent’s closing price on October 23, 2025 and November 6, 2025. Did trading volume support a strong directional breakout?","Tencent’s closing price increased from 633.0 to 644.0 between October 23 and November 6, 2025. Trading volume remained relatively stable, suggesting a gradual upward move rather than a strong breakout.","Tencent’s closing price on October 23, 2025, was 633.0 HKD with a trading volume of 12,419,348 shares ([Source 1]). However, the retrieved context does not provide Tencent’s closing price or volume data for November 6, 2025. \n\nWithout the November 6 data, it is not possible to compare the clos...",all_prices_clean.csv
5,"How does the April 17, 2026 market intelligence reporting connect the Strait of Hormuz blockade to Asian fuel supply concerns?","The reports explain that the naval blockade disrupted Middle Eastern energy flows, forcing Asian refiners in Japan and South Korea to source additional US crude oil supplies in order to avoid fuel shortages.","The April 17, 2026 market intelligence reporting highlights that the blockade of the Strait of Hormuz by Iran has severely disrupted Middle Eastern oil supply, causing Asian fuel refiners—particularly in countries like Japan and South Korea—to increasingly rely on US crud

Evaluating optimized-prompt case 1 (1/8)
Evaluating optimized-prompt case 4 (2/8)
Evaluating optimized-prompt case 7 (3/8)
Evaluating optimized-prompt case 8 (4/8)
Evaluating optimized-prompt case 13 (5/8)
Evaluating optimized-prompt case 16 (6/8)
Evaluating optimized-prompt case 17 (7/8)
Evaluating optimized-prompt case 20 (8/8)
Optimized prompt evaluation summary


,metric,average_score,evaluated_cases,successful_cases,failed_or_error_cases
2,Broker Actionability,0.874223,8,8,0
1,Answer Relevance,0.864735,8,7,0
4,Faithfulness / Groundedness,0.756172,8,8,0
0,Answer Correctness,0.639538,8,5,0
3,Context Relevance,0.478564,8,3,0


Optimized prompt scores by benchmark case


metric,case_id,Answer Correctness,Answer Relevance,Broker Actionability,Context Relevance,Faithfulness / Groundedness
0,1,0.900000,0.900000,0.781757,0.202298,0.712369
1,4,0.404215,0.410353,0.679365,0.349327,0.653210
2,7,1.000000,1.000000,1.000000,0.906009,0.822270
3,8,1.000000,1.000000,1.000000,0.992414,0.735739
4,13,0.026894,0.881757,0.988080,0.190465,0.811920
5,16,1.000000,0.977730,0.801814,0.811920,0.781757
6,17,0.785195,0.950000,0.843782,0.294322,0.564188
7,20,0.000000,0.798037,0.898988,0.081757,0.967918


## 2.4 Compare optimized prompt with the baseline prompt


In [22]:
# Objective: compare prompts at metric, case, and overall levels, then apply
# quality gates so a small average gain cannot hide a critical regression.
baseline_comparison_df = baseline_metric_summary_df.rename(
    columns={
        "average_score": "baseline_average_score",
        "evaluated_cases": "baseline_evaluated_cases",
        "successful_cases": "baseline_successful_cases",
        "failed_or_error_cases": "baseline_error_cases",
    }
)

optimized_comparison_df = optimized_metric_summary_df.rename(
    columns={
        "average_score": "optimized_average_score",
        "evaluated_cases": "optimized_evaluated_cases",
        "successful_cases": "optimized_successful_cases",
        "failed_or_error_cases": "optimized_error_cases",
    }
)

prompt_comparison_df = (
    baseline_comparison_df
    .merge(optimized_comparison_df, on="metric", how="outer")
)

prompt_comparison_df["score_delta"] = (
    prompt_comparison_df["optimized_average_score"]
    - prompt_comparison_df["baseline_average_score"]
)

nonzero_baseline_scores = prompt_comparison_df[
    "baseline_average_score"
].replace(0, pd.NA)

prompt_comparison_df["improvement_pct"] = (
    prompt_comparison_df["score_delta"]
    .div(nonzero_baseline_scores)
    .mul(100)
)


def select_winner(row, tolerance=1e-9):
    baseline_score = row["baseline_average_score"]
    optimized_score = row["optimized_average_score"]

    if pd.isna(baseline_score) or pd.isna(optimized_score):
        return "Unavailable"
    if abs(optimized_score - baseline_score) <= tolerance:
        return "Tie"
    if optimized_score > baseline_score:
        return "Optimized"
    return "Baseline"


prompt_comparison_df["winner"] = prompt_comparison_df.apply(
    select_winner,
    axis=1,
)

prompt_comparison_df = prompt_comparison_df.sort_values(
    "score_delta",
    ascending=False,
).reset_index(drop=True)

print("Baseline vs. optimized prompt — metric summary")
display(
    prompt_comparison_df[
        [
            "metric",
            "baseline_average_score",
            "optimized_average_score",
            "score_delta",
            "improvement_pct",
            "winner",
            "baseline_error_cases",
            "optimized_error_cases",
        ]
    ].round(4)
)


# Compare both prompts for each benchmark case and metric.
baseline_case_scores_df = baseline_evaluation_df[
    ["case_id", "question", "metric", "score"]
].rename(columns={"score": "baseline_score"})

optimized_case_scores_df = optimized_evaluation_df[
    ["case_id", "question", "metric", "score"]
].rename(columns={"score": "optimized_score"})

case_level_comparison_df = baseline_case_scores_df.merge(
    optimized_case_scores_df,
    on=["case_id", "question", "metric"],
    how="inner",
)

case_level_comparison_df["score_delta"] = (
    case_level_comparison_df["optimized_score"]
    - case_level_comparison_df["baseline_score"]
)

case_level_comparison_df["outcome"] = case_level_comparison_df.apply(
    lambda row: select_winner(
        pd.Series({
            "baseline_average_score": row["baseline_score"],
            "optimized_average_score": row["optimized_score"],
        })
    ),
    axis=1,
)

case_level_outcomes_df = (
    case_level_comparison_df
    .groupby(["metric", "outcome"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

print("Per-case comparison counts")
display(case_level_outcomes_df)


# Create one compact overall comparison across all questions and metrics.
def evaluation_success_rate(evaluation_df):
    valid_success_values = evaluation_df["success"].dropna()
    if valid_success_values.empty:
        return None
    return valid_success_values.astype(bool).mean()


overall_prompt_comparison_df = pd.DataFrame([
    {
        "prompt": "Baseline",
        "average_score": baseline_evaluation_df["score"].mean(),
        "success_rate": evaluation_success_rate(
            baseline_evaluation_df
        ),
        "evaluated_scores": baseline_evaluation_df["score"].count(),
        "error_count": baseline_evaluation_df["error"].notna().sum(),
    },
    {
        "prompt": "Optimized",
        "average_score": optimized_evaluation_df["score"].mean(),
        "success_rate": evaluation_success_rate(
            optimized_evaluation_df
        ),
        "evaluated_scores": optimized_evaluation_df["score"].count(),
        "error_count": optimized_evaluation_df["error"].notna().sum(),
    },
])

print("Overall prompt comparison")
display(overall_prompt_comparison_df.round(4))

def normalize_prompt(text):
    return " ".join(text.split())

prompt_was_modified = (
    normalize_prompt(optimized_prompt_text)
    != normalize_prompt(seed_prompt_text)
)
optimized_metric_wins = (
    prompt_comparison_df["winner"] == "Optimized"
).sum()
baseline_metric_wins = (
    prompt_comparison_df["winner"] == "Baseline"
).sum()
metric_ties = (prompt_comparison_df["winner"] == "Tie").sum()

print(f"Did GEPA modify the seed prompt? {prompt_was_modified}")
print(
    "Metric wins — "
    f"Optimized: {optimized_metric_wins}, "
    f"Baseline: {baseline_metric_wins}, "
    f"Ties: {metric_ties}"
)

# Weighted scoring reflects the higher cost of factual or grounding failures.
METRIC_WEIGHTS = {
    "Answer Correctness": 0.35,
    "Faithfulness / Groundedness": 0.25,
    "Context Relevance": 0.15,
    "Answer Relevance": 0.10,
    "Broker Actionability": 0.15,
}

def weighted_metric_score(summary_df):
    scores = summary_df.set_index("metric")["average_score"]
    return sum(scores.get(metric, 0) * weight for metric, weight in METRIC_WEIGHTS.items())

baseline_weighted = weighted_metric_score(baseline_metric_summary_df)
optimized_weighted = weighted_metric_score(optimized_metric_summary_df)
baseline_scores = baseline_metric_summary_df.set_index("metric")["average_score"]
optimized_scores = optimized_metric_summary_df.set_index("metric")["average_score"]
baseline_success_rate = evaluation_success_rate(baseline_evaluation_df)
optimized_success_rate = evaluation_success_rate(optimized_evaluation_df)

QUALITY_POLICY = {"minimum_gain": 0.01, "critical_tolerance": 0.02}
quality_gates = {
    "weighted_score_gain": optimized_weighted >= baseline_weighted + QUALITY_POLICY["minimum_gain"],
    "correctness_non_regression": optimized_scores["Answer Correctness"] >= baseline_scores["Answer Correctness"] - QUALITY_POLICY["critical_tolerance"],
    "faithfulness_non_regression": optimized_scores["Faithfulness / Groundedness"] >= baseline_scores["Faithfulness / Groundedness"] - QUALITY_POLICY["critical_tolerance"],
    "success_rate_non_regression": optimized_success_rate >= baseline_success_rate,
    "no_additional_errors": optimized_evaluation_df["error"].notna().sum() <= baseline_evaluation_df["error"].notna().sum(),
}
prompt_decision = "APPROVE" if all(quality_gates.values()) else "REJECT"
quality_gate_df = pd.DataFrame([
    {"gate": gate, "passed": passed}
    for gate, passed in quality_gates.items()
])

print(f"Weighted score — Baseline: {baseline_weighted:.4f} | Optimized: {optimized_weighted:.4f}")
print(f"Final prompt decision: {prompt_decision}")
display(quality_gate_df)


Baseline vs. optimized prompt — metric summary


,metric,baseline_average_score,optimized_average_score,score_delta,improvement_pct,winner,baseline_error_cases,optimized_error_cases
0,Broker Actionability,0.7952,0.8742,0.0790,9.9392,Optimized,0,0
1,Faithfulness / Groundedness,0.7046,0.7562,0.0516,7.3182,Optimized,0,0
2,Answer Relevance,0.8563,0.8647,0.0084,0.9865,Optimized,0,0
3,Context Relevance,0.4814,0.4786,-0.0028,-0.5812,Baseline,0,0
4,Answer Correctness,0.6582,0.6395,-0.0186,-2.8293,Baseline,0,0


Per-case comparison counts


outcome,metric,Baseline,Optimized,Tie
0,Answer Correctness,3,3,2
1,Answer Relevance,3,3,2
2,Broker Actionability,2,4,2
3,Context Relevance,2,1,5
4,Faithfulness / Groundedness,3,5,0


Overall prompt comparison


,prompt,average_score,success_rate,evaluated_scores,error_count
0,Baseline,0.6991,0.725,40,0
1,Optimized,0.7226,0.775,40,0


Did GEPA modify the seed prompt? True
Metric wins — Optimized: 3, Baseline: 2, Ties: 0
Weighted score — Baseline: 0.6836 | Optimized: 0.7023
Final prompt decision: APPROVE


,gate,passed
0,weighted_score_gain,True
1,correctness_non_regression,True
2,faithfulness_non_regression,True
3,success_rate_non_regression,True
4,no_additional_errors,True


# Section 3: RAG Configuration Tuning and Evaluation


## 3.1 Define the RAG configurations

In [23]:
# Write your code here

## 3.2 Create a runtime answering function

In [24]:
# Write your code here

## 3.3 Evaluate one configuration on a benchmark set

In [25]:
# Write your code here

## 3.4 Run all configurations on the test set

In [26]:
# Write your code here

## 3.5 Compare configurations

In [27]:
# Write your code here

## 3.6 Select the best overall approach

In [28]:
# Write your code here

## 3.7 Save the tuning results


In [29]:
# Write your code here

# Section 4: Final RAG Inference Pipeline with the Optimized Prompt and Best RAG Configuration

## 4.1 Define the final inference helper

In [30]:
# Write your code here

## 4.2 Define the final test cases

In [31]:
final_test_cases = [
    {
        "id": 1,
        "question": (
            "Compare the total revenue and net income reported by Apple, Amazon, "
            "and Alphabet in their latest 10-Q filings. Which company grew fastest "
            "year-over-year?"
        )
    },
    {
        "id": 2,
        "question": (
            "Based on this week's news sentiment around Amazon, would you classify "
            "the current signal as bullish or bearish, and does the price data support that?"
        )
    },
    {
        "id": 3,
        "question": (
            "Should I invest in Apple right now? I heard Tim Cook is stepping down — "
            "is that a red flag? How's the stock been doing, and is there anything I should "
            "be worried about with the company's financials or risks?"
        )
    },
    {
        "id": 4,
        "question": (
            "Is now a good time to buy Bitcoin? News is saying it just crossed $78k — "
            "am I too late? What's the price trend looked like over the last few months, "
            "and what's driving the current rally?"
        )
    },
    {
        "id": 5,
        "question": (
            "I want to invest in AI. Out of Google, Amazon, and Apple, which one looks "
            "like the best bet right now? How are they performing, what are they saying "
            "about their AI business, and what's the buzz around them?"
        )
    },
]

## 4.3 Run the final test cases

In [32]:
# Write your code here

# Conclusion

## Actionable Insights:

- WRITE INSIGHTS HERE

## Recommendations:

- WRITE RECOMMENDATIONS HERE

<font size=6>Power Ahead!</font>
___